# PARTIE A : EXPLORATION

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("ventes_ecommerce.csv", parse_dates=['date_commande'])
df.head()

,id_commande,date_commande,client_id,ville_client,categorie_produit,nom_produit,quantite,prix_unitaire,remise_pourcentage,mode_paiement,canal_vente,delai_livraison_jours,note_satisfaction,statut_commande
0,104746,2025-01-28,1412,Lille,Vetements,Baskets,4,40.30,5,PayPal,Site web,3.0,4.0,Livree
1,103100,2025-08-13,2494,Lille,Maison,Tapis,2,48.13,20,CB,Site web,11.0,5.0,Livree
2,103690,2024-06-01,1381,Bordeaux,Maison,Etagere,1,46.61,10,CB,Magasin,3.0,NaN,Livree
3,102454,2024-04-16,2017,Toulouse,Maison,Set de cuisine,1,49.40,0,CB,Application mobile,14.0,3.0,Livree
4,100437,2025-02-14,1074,Lyon,Electronique,Tablette,1,197.30,0,CB,Magasin,8.0,5.0,Livree


### Afficher la forme 

In [3]:
df.shape

(5025, 14)

### Types des colnnes

In [4]:
df.dtypes

id_commande                       int64
date_commande            datetime64[ns]
client_id                         int64
ville_client                     object
categorie_produit                object
nom_produit                      object
quantite                          int64
prix_unitaire                   float64
remise_pourcentage                int64
mode_paiement                    object
canal_vente                      object
delai_livraison_jours           float64
note_satisfaction               float64
statut_commande                  object
dtype: object

### Statistiques descriptives des connes

In [5]:
df.describe()

,id_commande,date_commande,client_id,quantite,prix_unitaire,remise_pourcentage,delai_livraison_jours,note_satisfaction
count,5025.000000,5025,5025.000000,5025.000000,4925.000000,5025.000000,4974.000000,4530.000000
mean,102500.722587,2024-12-30 06:31:09.850746368,1750.779502,2.362189,97.144211,10.280597,7.543024,3.901766
min,100000.000000,2024-01-01 00:00:00,1000.000000,1.000000,3.000000,0.000000,1.000000,1.000000
25%,101251.000000,2024-06-30 00:00:00,1379.000000,1.000000,25.790000,0.000000,4.000000,3.000000
50%,102502.000000,2024-12-30 00:00:00,1742.000000,2.000000,44.390000,10.000000,8.000000,4.000000
75%,103751.000000,2025-07-02 00:00:00,2123.000000,3.000000,111.930000,20.000000,11.000000,5.000000
max,104999.000000,2025-12-30 00:00:00,2499.000000,5.000000,16153.500000,30.000000,14.000000,5.000000
std,1442.695335,NaN,431.837723,1.387846,260.750370,10.367975,4.049669,1.105725


### Afficher le nombre de valeurs uniques

In [6]:
one = ['categorie_produit', 'canal_vente', 'statut_commande']
df[one].nunique()

categorie_produit    6
canal_vente          3
statut_commande      4
dtype: int64

### Répartition des commandes par statut

In [7]:
df.groupby('statut_commande')['quantite'].value_counts()

statut_commande  quantite
Annulee          1            157
                 2            100
                 3             64
                 5             41
                 4             39
En cours         1            139
                 2            110
                 5             59
                 4             52
                 3             39
Livree           1           1452
                 2           1004
                 3            510
                 4            498
                 5            455
Retournee        1            111
                 2             80
                 3             44
                 5             37
                 4             34
Name: count, dtype: int64

# PARTIE B : NETTOYAGE 

### Nombres de valeurs manquantes par colonne

In [8]:
df.isna().sum()

id_commande                0
date_commande              0
client_id                  0
ville_client              75
categorie_produit          0
nom_produit                0
quantite                   0
prix_unitaire            100
remise_pourcentage         0
mode_paiement              0
canal_vente                0
delai_livraison_jours     51
note_satisfaction        495
statut_commande            0
dtype: int64

### Nombres de lignes dupliquées

In [9]:
print(df.duplicated().sum())

25


### Suppression des doublons

In [10]:
df = df.drop_duplicates()
print(df.duplicated().sum())

0


###  Remplacons les NaN de prix_unitaire par la mediane de la categorie_produit correspondante

(indice : groupby('categorie_produit')['prix_unitaire'].transform('median'))

In [11]:
df['prix_unitaire'] = df['prix_unitaire'].fillna(
    df.groupby('categorie_produit')['prix_unitaire'].transform('median')
)

### Remplace les NaN de ville_client par 'Inconnue', et ceux de delai_livraison_jours par la mediane generale de la colonne

In [12]:
df['ville_client'] = df['ville_client'].fillna('Inconnues')

In [13]:
delai_median = df['delai_livraison_jours'].median()

df['delai_livraison_jours'] = df['delai_livraison_jours'].fillna('delai_median')

### Vérification 

In [14]:
df.isna().sum()

id_commande                0
date_commande              0
client_id                  0
ville_client               0
categorie_produit          0
nom_produit                0
quantite                   0
prix_unitaire              0
remise_pourcentage         0
mode_paiement              0
canal_vente                0
delai_livraison_jours      0
note_satisfaction        495
statut_commande            0
dtype: int64

### Détection des prix aberrants

In [15]:
mediane_cat = df.groupby('categorie_produit')['prix_unitaire'].transform('median')
outliers = df[df['prix_unitaire'] > 5 * mediane_cat]
print(outliers)

      id_commande date_commande  client_id ville_client categorie_produit  \
963        103292    2024-07-13       2324       Nantes            Maison   
2225       104956    2024-06-06       2122     Toulouse             Sport   
2431       103635    2024-10-12       1133     Bordeaux      Electronique   
3341       101015    2024-04-20       1334         Lyon            Beaute   
4053       103424    2024-06-12       1255         Lyon            Maison   
4898       103098    2024-12-03       1329     Toulouse             Sport   

             nom_produit  quantite  prix_unitaire  remise_pourcentage  \
963                Tapis         5         1850.5                  15   
2225    Montre connectee         1         2829.0                  15   
2431  Enceinte bluetooth         4        16153.5                   0   
3341           Shampoing         1          530.5                  30   
4053             Coussin         2         2035.5                  30   
4898    Montre connect

### Remplacons ensuite ces valeurs par la mediane de la categorie

In [16]:
df.loc[outliers.index, 'prix_unitaire'] = mediane_cat[outliers.index]

# PARTIE C : FILTRAGE ET TRI

### Création d'un sous ensemble sur le statut de commande

In [17]:
df_livrees = df[df['statut_commande'] == 'Livree']
df_livrees['statut_commande']

0       Livree
1       Livree
2       Livree
3       Livree
4       Livree
         ...  
5020    Livree
5021    Livree
5022    Livree
5023    Livree
5024    Livree
Name: statut_commande, Length: 3898, dtype: object

### Dans df_livrees, trouve les 10 commandes ayant genere le plus gros montant (quantite x prix_unitaire), triees par ordre decroissant

In [18]:
top10 = df_livrees.assign(
    montant=df_livrees['quantite'] * df_livrees['prix_unitaire']
).sort_values('montant', ascending=False).head(10)
print(top10)

      id_commande date_commande  client_id ville_client categorie_produit  \
3870       104653    2024-06-26       2314        Paris      Electronique   
3823       102627    2024-01-30       1939     Toulouse      Electronique   
3283       104881    2025-05-15       2407     Bordeaux      Electronique   
2530       100223    2025-05-13       2351        Paris      Electronique   
1692       100532    2025-02-20       1094    Marseille      Electronique   
3156       101864    2025-09-03       1087       Nantes      Electronique   
1245       102381    2024-09-17       2118        Lille      Electronique   
2383       101424    2025-01-31       2471        Paris      Electronique   
1139       102285    2024-11-08       2410        Paris      Electronique   
3337       103947    2024-01-22       1484         Lyon      Electronique   

              nom_produit  quantite  prix_unitaire  remise_pourcentage  \
3870   Enceinte bluetooth         5         410.02                  20   
3823

### Filtre les commandes de la categorie 'Electronique' passees via 'Site web' ET ayant une note_satisfaction superieure ou egale a 4

In [19]:
filtre1 = df[
    (df['categorie_produit'] == 'Electronique') &
    (df['canal_vente'] == 'Site web') &
    (df['note_satisfaction'] >= 4)
]
print(filtre1)

      id_commande date_commande  client_id ville_client categorie_produit  \
29         104979    2024-08-26       1118         Lyon      Electronique   
49         101416    2024-04-26       2404       Nantes      Electronique   
50         102993    2024-02-04       1605        Lille      Electronique   
68         104259    2025-01-11       1838        Lille      Electronique   
77         100111    2025-02-07       1884   Strasbourg      Electronique   
...           ...           ...        ...          ...               ...   
5004       104019    2024-09-17       2366     Toulouse      Electronique   
5006       103388    2025-05-06       1880   Strasbourg      Electronique   
5007       102306    2024-08-01       1507     Toulouse      Electronique   
5010       101498    2025-03-13       1748    Marseille      Electronique   
5022       101667    2025-06-22       2024        Paris      Electronique   

             nom_produit  quantite  prix_unitaire  remise_pourcentage  \
29

### Trouve toutes les commandes dont le delai_livraison_jours depasse 10 jours ET dont le statut est 'Livree' — ce sont des livraisons lentes malgre tout abouties

In [20]:
print(df['delai_livraison_jours'].dtype) 

object


In [21]:
df['delai_livraison_jours'] = pd.to_numeric(df['delai_livraison_jours'], errors='coerce')

livraisons_lentes = df[
    (df['delai_livraison_jours'] > 10) & (df['statut_commande'] == 'Livree')
]
print(livraisons_lentes)

      id_commande date_commande  client_id ville_client categorie_produit  \
1          103100    2025-08-13       2494        Lille            Maison   
3          102454    2024-04-16       2017     Toulouse            Maison   
17         104149    2025-07-23       1608     Toulouse      Electronique   
24         101808    2024-02-10       1566   Strasbourg             Sport   
29         104979    2024-08-26       1118         Lyon      Electronique   
...           ...           ...        ...          ...               ...   
5017       102923    2025-07-02       1486     Bordeaux            Maison   
5018       102707    2025-02-18       1508    Inconnues             Sport   
5020       100789    2024-09-24       1342         Lyon      Electronique   
5023       103321    2024-07-07       1767     Toulouse         Vetements   
5024       101688    2024-06-25       1346         Lyon             Sport   

           nom_produit  quantite  prix_unitaire  remise_pourcentage  \
1   

# PARTIE D : AGREGATION

### Cree une colonne montant_total = quantite x prix_unitaire x (1 - remise_pourcentage / 100)

In [22]:
df['montant_total'] = df['quantite'] * df['prix_unitaire'] * (1 - (df['remise_pourcentage'] / 100))

In [23]:
df['montant_total']

0       153.140
1        77.008
2        41.949
3        49.400
4       197.300
         ...   
5020    952.770
5021     20.060
5022    230.472
5023     64.152
5024    155.520
Name: montant_total, Length: 5000, dtype: float64

### Calcule le chiffre d'affaires total (somme de montant_total) sur df_livrees uniquement — seules les commandes livrees comptent comme revenu reel

In [24]:
df_livrees = df[df['statut_commande'] == 'Livree']
df_livrees.head()

,id_commande,date_commande,client_id,ville_client,categorie_produit,nom_produit,quantite,prix_unitaire,remise_pourcentage,mode_paiement,canal_vente,delai_livraison_jours,note_satisfaction,statut_commande,montant_total
0,104746,2025-01-28,1412,Lille,Vetements,Baskets,4,40.30,5,PayPal,Site web,3.0,4.0,Livree,153.140
1,103100,2025-08-13,2494,Lille,Maison,Tapis,2,48.13,20,CB,Site web,11.0,5.0,Livree,77.008
2,103690,2024-06-01,1381,Bordeaux,Maison,Etagere,1,46.61,10,CB,Magasin,3.0,NaN,Livree,41.949
3,102454,2024-04-16,2017,Toulouse,Maison,Set de cuisine,1,49.40,0,CB,Application mobile,14.0,3.0,Livree,49.400
4,100437,2025-02-14,1074,Lyon,Electronique,Tablette,1,197.30,0,CB,Magasin,8.0,5.0,Livree,197.300


In [25]:
somme_montant_total_livrees = df_livrees['montant_total'].sum()

In [26]:
print(somme_montant_total_livrees)

748812.8535


### Calcule le chiffre d'affaires total par categorie_produit, triee du plus rentable au moins rentable

In [27]:
chiffre_aff = df_livrees.groupby('categorie_produit')['montant_total'].sum().sort_values(ascending = False)

In [28]:
chiffre_aff

categorie_produit
Electronique    515970.7360
Sport            75641.1735
Maison           70271.5175
Vetements        54991.3385
Beaute           20537.5645
Livres           11400.5235
Name: montant_total, dtype: float64

### Pour chaque canal_vente, calcule en une seule instruction .agg() : nombre de commandes, montant moyen, et note de satisfaction moyenne 

In [29]:
resume_canal = df.groupby('canal_vente').agg(
    nb_commandes = ('quantite', 'count'),
    montant_moyen = ('montant_total', 'mean'),
    satisfaction_moyen = ('note_satisfaction', 'mean')
)
print(resume_canal)

                    nb_commandes  montant_moyen  satisfaction_moyen
canal_vente                                                        
Application mobile          1473     192.184956            3.904618
Magasin                     1007     189.043268            3.899012
Site web                    2520     195.309204            3.898812


### Calcule le taux d'annulation par categorie_produit (indice : nombre de commandes 'Annulee' divise par le nombre total de commandes, pour chaque categorie — tu peux passer par une fonction lambda dans agg, ou calculer en deux etapes)

In [30]:
taux_annulation = df.groupby('categorie_produit')['statut_commande'].apply(
    lambda x: (x == 'Annulee').sum() / len(x)
)
print(taux_annulation)

categorie_produit
Beaute          0.087986
Electronique    0.076804
Livres          0.101626
Maison          0.083987
Sport           0.057065
Vetements       0.083083
Name: statut_commande, dtype: float64


# PARTIE E : PIVOT_TABLE 

###  Cree un pivot_table du chiffre d'affaires (montant_total) avec categorie_produit en index et canal_vente en colonnes, aggfunc='sum'

In [31]:
chiffre_affaires = df.pivot_table(values = 'montant_total', index='categorie_produit', columns='canal_vente', aggfunc='sum')
print(chiffre_affaires)

canal_vente        Application mobile      Magasin     Site web
categorie_produit                                              
Beaute                     6935.69350    5457.7290   13262.7355
Electronique             195147.00900  133266.5755  341428.9180
Livres                     5245.11325    3314.4845    6934.2865
Maison                    25780.92800   16823.9140   46571.8395
Sport                     27533.16600   16994.5210   48598.8275
Vetements                 22446.53100   14509.3470   35382.5875


### Cree un pivot_table de la note de satisfaction moyenne, avec ville_client en index et categorie_produit en colonnes

In [32]:
satisfaction_mean = df.pivot_table(values='note_satisfaction', index='ville_client', columns='categorie_produit', aggfunc='mean')
print(satisfaction_mean)

categorie_produit    Beaute  Electronique    Livres    Maison     Sport  \
ville_client                                                              
Bordeaux           3.779412      3.991803  4.018182  4.010101  3.717391   
Inconnues          3.916667      4.375000  5.000000  3.454545  4.222222   
Lille              3.934211      4.039216  3.745763  3.745283  3.982456   
Lyon               3.983871      4.033784  4.057692  3.979381  3.863636   
Marseille          3.965517      3.858108  3.769231  3.965116  4.010638   
Nantes             3.822785      4.050000  3.711111  3.936170  3.882353   
Paris              4.180328      3.944056  4.101695  3.853448  3.761905   
Strasbourg         3.776316      3.761589  4.090909  3.663043  3.740260   
Toulouse           3.891304      3.780822  3.919355  3.858696  3.743902   

categorie_produit  Vetements  
ville_client                  
Bordeaux            4.071429  
Inconnues           4.062500  
Lille               4.017241  
Lyon               

###  Dans le premier pivot_table (question E1), trouve quelle combinaison categorie/canal genere le plus gros chiffre d'affaires. Indice : .stack() transforme le tableau en serie, puis idxmax() donne l'index du maximum

In [33]:
maxim = chiffre_affaires.stack()
maxim.idxmax()

('Electronique', 'Site web')

# PARTIE F : APPLY / LAMBDA

### Cree une colonne tranche_montant qui vaut 'Petit' si montant_total < 30, 'Moyen' si entre 30 et 150, sinon 'Gros', en utilisant apply avec une fonction lambda

In [39]:
df['tranche_montant'] = df['montant_total'].apply(lambda x : 'Petit' if x < 30 else('Moyen' if x >= 30 and x <= 150 else 'Gros'))

In [40]:
df.head()

,id_commande,date_commande,client_id,ville_client,categorie_produit,nom_produit,quantite,prix_unitaire,remise_pourcentage,mode_paiement,canal_vente,delai_livraison_jours,note_satisfaction,statut_commande,montant_total,tranche_montant
0,104746,2025-01-28,1412,Lille,Vetements,Baskets,4,40.30,5,PayPal,Site web,3.0,4.0,Livree,153.140,Gros
1,103100,2025-08-13,2494,Lille,Maison,Tapis,2,48.13,20,CB,Site web,11.0,5.0,Livree,77.008,Moyen
2,103690,2024-06-01,1381,Bordeaux,Maison,Etagere,1,46.61,10,CB,Magasin,3.0,NaN,Livree,41.949,Moyen
3,102454,2024-04-16,2017,Toulouse,Maison,Set de cuisine,1,49.40,0,CB,Application mobile,14.0,3.0,Livree,49.400,Moyen
4,100437,2025-02-14,1074,Lyon,Electronique,Tablette,1,197.30,0,CB,Magasin,8.0,5.0,Livree,197.300,Gros


### Cree une colonne panier qui vaut 'Multi-articles' si quantite > 1, sinon 'Article unique'

In [41]:
df['panier'] = df['quantite'].apply(lambda x : 'Multi articles' if x > 1 else 'Article unique')

In [42]:
df.head()

,id_commande,date_commande,client_id,ville_client,categorie_produit,nom_produit,quantite,prix_unitaire,remise_pourcentage,mode_paiement,canal_vente,delai_livraison_jours,note_satisfaction,statut_commande,montant_total,tranche_montant,panier
0,104746,2025-01-28,1412,Lille,Vetements,Baskets,4,40.30,5,PayPal,Site web,3.0,4.0,Livree,153.140,Gros,Multi articles
1,103100,2025-08-13,2494,Lille,Maison,Tapis,2,48.13,20,CB,Site web,11.0,5.0,Livree,77.008,Moyen,Multi articles
2,103690,2024-06-01,1381,Bordeaux,Maison,Etagere,1,46.61,10,CB,Magasin,3.0,NaN,Livree,41.949,Moyen,Article unique
3,102454,2024-04-16,2017,Toulouse,Maison,Set de cuisine,1,49.40,0,CB,Application mobile,14.0,3.0,Livree,49.400,Moyen,Article unique
4,100437,2025-02-14,1074,Lyon,Electronique,Tablette,1,197.30,0,CB,Magasin,8.0,5.0,Livree,197.300,Gros,Article unique


### Calcule le montant total genere par tranche_montant avec un groupby

In [43]:
mont_tot_gen = df.groupby('tranche_montant')['montant_total'].sum()
mont_tot_gen

tranche_montant
Gros     773282.36750
Moyen    175711.87500
Petit     16639.96375
Name: montant_total, dtype: float64

### Cree une fonction Python classique (avec def) qui calcule une remise 'fidelite' fictive : si le client_id est un nombre pair, remise de 2 euros de plus sur le montant_total, sinon rien. Applique-la avec apply pour creer la colonne montant_final

In [46]:
def remise_fidelite(frame):
    
    if frame['client_id'] % 2 == 0:
        return frame['montant_total'] - 2
    return frame['montant_total']

    
df['montant_final'] = df.apply(remise_fidelite, axis=1)
df[['client_id', 'montant_total', 'montant_final']].head()

,client_id,montant_total,montant_final
0,1412,153.140,151.140
1,2494,77.008,75.008
2,1381,41.949,41.949
3,2017,49.400,49.400
4,1074,197.300,195.300


# PARTIE G : Date et Texte

### A partir de date_commande, cree deux colonnes : annee et mois (indice : .dt.year, .dt.month)

In [57]:
df['annee'] = df['date_commande'].dt.year
df['mois'] = df['date_commande'].dt.month

In [58]:
df[['annee', 'mois']].head()

,annee,mois
0,2025,1
1,2025,8
2,2024,6
3,2024,4
4,2025,2


### Cree une colonne jour_semaine avec le nom du jour de la semaine (indice : .dt.day_name())

In [59]:
df['jour_semaine'] = df['date_commande'].dt.day_name()

In [60]:
df[['annee', 'mois', 'jour_semaine']].head()

,annee,mois,jour_semaine
0,2025,1,Tuesday
1,2025,8,Wednesday
2,2024,6,Saturday
3,2024,4,Tuesday
4,2025,2,Friday


### Calcule le chiffre d'affaires total par mois (groupby sur la colonne mois ou sur date_commande.dt.to_period('M')). Quel mois a genere le plus de chiffre d'affaires

In [62]:
ca_mois = df.groupby('mois')['montant_total'].sum()
ca_mois

mois
1     83146.88350
2     71558.99500
3     79533.90650
4     83968.06150
5     95034.19550
6     74746.56850
7     83099.20650
8     81003.30375
9     75301.11200
10    77057.21900
11    74557.43500
12    86627.31950
Name: montant_total, dtype: float64

In [64]:
ca_mois.idxmax()

np.int32(5)

Le mois de Mai(5) a généré le plus de CA.

###  Trouve quel jour_semaine recoit le plus de commandes en moyenne (value_counts sur jour_semaine)

In [74]:
cmd_jour = df['jour_semaine'].value_counts()
cmd_jour

jour_semaine
Saturday     746
Tuesday      743
Wednesday    721
Thursday     705
Monday       700
Friday       699
Sunday       686
Name: count, dtype: int64

Le Samedi recoit le plus de commmandes en moyenne.

### En utilisant .str sur nom_produit, trouve toutes les commandes dont le nom du produit contient le mot 'connectee' (categorie Sport) - indice : .str.contains('connectee', case=False)

In [76]:
df.head()

,id_commande,date_commande,client_id,ville_client,categorie_produit,nom_produit,quantite,prix_unitaire,remise_pourcentage,mode_paiement,...,delai_livraison_jours,note_satisfaction,statut_commande,montant_total,tranche_montant,panier,montant_final,annee,mois,jour_semaine
0,104746,2025-01-28,1412,Lille,Vetements,Baskets,4,40.30,5,PayPal,...,3.0,4.0,Livree,153.140,Gros,Multi articles,151.140,2025,1,Tuesday
1,103100,2025-08-13,2494,Lille,Maison,Tapis,2,48.13,20,CB,...,11.0,5.0,Livree,77.008,Moyen,Multi articles,75.008,2025,8,Wednesday
2,103690,2024-06-01,1381,Bordeaux,Maison,Etagere,1,46.61,10,CB,...,3.0,NaN,Livree,41.949,Moyen,Article unique,41.949,2024,6,Saturday
3,102454,2024-04-16,2017,Toulouse,Maison,Set de cuisine,1,49.40,0,CB,...,14.0,3.0,Livree,49.400,Moyen,Article unique,49.400,2024,4,Tuesday
4,100437,2025-02-14,1074,Lyon,Electronique,Tablette,1,197.30,0,CB,...,8.0,5.0,Livree,197.300,Gros,Article unique,195.300,2025,2,Friday


In [81]:
connect = df[df['nom_produit'].str.contains('connectee', case = False)]
connect

,id_commande,date_commande,client_id,ville_client,categorie_produit,nom_produit,quantite,prix_unitaire,remise_pourcentage,mode_paiement,...,delai_livraison_jours,note_satisfaction,statut_commande,montant_total,tranche_montant,panier,montant_final,annee,mois,jour_semaine
11,102426,2025-04-22,2356,Nantes,Sport,Montre connectee,2,19.81,0,CB,...,4.0,3.0,Livree,39.620,Moyen,Multi articles,37.620,2025,4,Tuesday
51,101155,2024-10-04,1034,Nantes,Sport,Montre connectee,2,53.69,5,Virement,...,6.0,4.0,Livree,102.011,Moyen,Multi articles,100.011,2024,10,Friday
60,102294,2025-09-01,1478,Toulouse,Sport,Montre connectee,1,54.82,10,PayPal,...,12.0,NaN,Livree,49.338,Moyen,Article unique,47.338,2025,9,Monday
62,101740,2025-07-24,1407,Toulouse,Sport,Montre connectee,5,58.24,10,Virement,...,14.0,5.0,Livree,262.080,Gros,Multi articles,262.080,2025,7,Thursday
66,103805,2024-08-30,2393,Lyon,Sport,Montre connectee,5,65.18,0,PayPal,...,8.0,4.0,Livree,325.900,Gros,Multi articles,325.900,2024,8,Friday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4976,102960,2024-05-13,2488,Lyon,Sport,Montre connectee,3,76.26,5,PayPal,...,14.0,3.0,Livree,217.341,Gros,Multi articles,215.341,2024,5,Monday
4985,102161,2025-04-22,1410,Toulouse,Sport,Montre connectee,3,72.07,30,PayPal,...,14.0,5.0,Livree,151.347,Gros,Multi articles,149.347,2025,4,Tuesday
5015,102528,2025-03-06,2331,Lille,Sport,Montre connectee,1,66.96,0,CB,...,2.0,3.0,Retournee,66.960,Moyen,Article unique,66.960,2025,3,Thursday
5018,102707,2025-02-18,1508,Inconnues,Sport,Montre connectee,4,79.80,5,PayPal,...,14.0,4.0,Livree,303.240,Gros,Multi articles,301.240,2025,2,Tuesday


# PARTIE H : NUMPY ET SYNTHESE STATISTIQUE

### Convertis la colonne montant_total en tableau NumPy et calcule avec numpy : la moyenne, l'ecart-type, et le 90e percentile (np.percentile)

In [82]:
import numpy as np

In [84]:
display(df['montant_total'])

0       153.140
1        77.008
2        41.949
3        49.400
4       197.300
         ...   
5020    952.770
5021     20.060
5022    230.472
5023     64.152
5024    155.520
Name: montant_total, Length: 5000, dtype: float64

In [85]:
montant_array = np.array(df['montant_total'])
montant_array

array([153.14 ,  77.008,  41.949, ..., 230.472,  64.152, 155.52 ])

In [89]:
print(np.mean(montant_array))
print(np.std(montant_array))
print(np.percentile(montant_array, 90))

193.12684125
272.48416171651326
525.1848000000001


### Calcule la correlation entre delai_livraison_jours et note_satisfaction (indice : df[['col1', 'col2']].corr()). Un delai plus long semble-t-il lie a une moins bonne satisfaction ?

**La corrélation est une mesure statistique qui indique dans quelle mesure deux variables évoluent ensemble.**

In [94]:
correlation = df[['delai_livraison_jours', 'note_satisfaction']].corr()
correlation

,delai_livraison_jours,note_satisfaction
delai_livraison_jours,1.000000,0.008616
note_satisfaction,0.008616,1.000000


# REDACTION FINALE 

**La catégorie la plus rentable dans ce dataset est : "Electronique"**

**Le canal de vente le plus rentable est : "Site Web" suivant le nombre de commandes, le montant moyen, et la note de satisfaction moyenne**

**Plus le delai de livraison est long, plus la note de satisfaction baisse**